# E09 - XỬ LÝ MẤT CÂN BẰNG LỚP & NGƯỠNG RIÊNG TỪNG KHÍA CẠNH (NGƯỜI 3)

**Vấn đề:**
1. Lớp Trung lập chỉ chiếm `19.6%` tập train; `class_weight='balanced'` có thể đẩy Recall lên nhưng làm sụt Precision.
2. Năm khía cạnh `asp_0..asp_4` có tần suất chênh nhau >2 lần, nên dùng chung 1 ngưỡng `0.5` chưa chắc tối ưu.

In [1]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand
    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'
    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            return os.path.dirname(os.path.dirname(p)), work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest
    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
FIG_DIR = os.path.join(WORK_DIR, 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')
print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT}')

Env: colab=False, kaggle=False | ROOT=D:\E-commerce


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from src.labels import ASPECT_GROUPS
from src.exp_log import log_experiment

df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
X_tr, X_dev = df_train['sentence_seg'].fillna(''), df_dev['sentence_seg'].fillna('')
y_tr_s, y_dev_s = df_train['sentiment'].values, df_dev['sentiment'].values
asp_cols = [f'asp_{i}' for i in range(5)]
Y_tr_a, Y_dev_a = df_train[asp_cols].values, df_dev[asp_cols].values
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]' 

## 1. SO SÁNH 4 CHIẾN LƯỢC CÂN BẰNG LỚP CHO SENTIMENT

In [3]:
s_cnt = df_train['sentiment'].value_counts().to_dict()
smooth_w = {c: float(np.sqrt(len(df_train) / (3.0 * cnt))) for c, cnt in s_cnt.items()}

# Random oversampling lop Trung lap (1) len bang lop lon nhat tren train
max_n = max(s_cnt.values())
df_neu = df_train[df_train['sentiment'] == 1]
df_os = pd.concat([df_train, df_neu.sample(n=max_n - len(df_neu), replace=True, random_state=42)], ignore_index=True)

sent_configs = [
    ('weight_none',    X_tr, y_tr_s, None),
    ('weight_balanced',X_tr, y_tr_s, 'balanced'),
    ('weight_smooth',  X_tr, y_tr_s, smooth_w),
    ('oversample_neu', df_os['sentence_seg'].fillna(''), df_os['sentiment'].values, None),
]

sent_rows = []
for name, xt, yt, cw in sent_configs:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight=cw, max_iter=1000, random_state=42)),
    ]).fit(xt, yt)
    pred = pipe.predict(X_dev)
    mf1 = f1_score(y_dev_s, pred, average='macro')
    acc = accuracy_score(y_dev_s, pred)
    neu_p = precision_score(y_dev_s, pred, labels=[1], average='macro')
    neu_r = recall_score(y_dev_s, pred, labels=[1], average='macro')
    neu_f1 = f1_score(y_dev_s, pred, labels=[1], average='macro')
    sent_rows.append({'strategy': name, 'dev_macro_f1': round(mf1, 4), 'acc': round(acc, 4),
                      'neu_P': round(neu_p, 4), 'neu_R': round(neu_r, 4), 'neu_F1': round(neu_f1, 4)})
    log_experiment('E09', f'sent_{name}', 'Class imbalance strategies (Sentiment)',
                   {'strategy': name}, dev_f1=mf1, notes=f'acc={acc:.4f}, neu_F1={neu_f1:.4f}, neu_P={neu_p:.4f}, neu_R={neu_r:.4f}')

pd.DataFrame(sent_rows)

,strategy,dev_macro_f1,acc,neu_P,neu_R,neu_F1
0,weight_none,0.7150,0.7982,0.6068,0.3287,0.4264
1,weight_balanced,0.7478,0.7856,0.5039,0.5972,0.5466
2,weight_smooth,0.7400,0.7973,0.5464,0.4630,0.5013
3,oversample_neu,0.7521,0.7919,0.5120,0.5926,0.5494


## 2. NGƯỠNG CHUNG `0.5` VS NGƯỠNG RIÊNG TỪNG KHÍA CẠNH (TUNE OOF TRÊN TRAIN)

In [4]:
pipe_asp = Pipeline([
    ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
    ('clf', OneVsRestClassifier(LogisticRegression(C=2.0, class_weight='balanced', max_iter=1000, random_state=42))),
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(pipe_asp, X_tr, Y_tr_a, cv=skf.split(X_tr, df_train['aspect_group']), method='predict_proba', n_jobs=1)

# Tim nguong rieng tren OOF cua tap TRAIN (tuyet doi khong nhin dev)
best_ths = []
grid = np.round(np.arange(0.30, 0.71, 0.02), 2)
for j in range(5):
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f = f1_score(Y_tr_a[:, j], (oof_probs[:, j] >= t).astype(int), zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), float(f)
    best_ths.append(best_t)
print('Per-aspect OOF tuned thresholds:', dict(zip(ASPECT_GROUPS, best_ths)))

pipe_asp.fit(X_tr, Y_tr_a)
dev_probs = pipe_asp.predict_proba(X_dev)
pred_05 = (dev_probs >= 0.5).astype(int)
pred_per = np.column_stack([(dev_probs[:, j] >= best_ths[j]).astype(int) for j in range(5)])

f1_05 = f1_score(Y_dev_a, pred_05, average='macro')
f1_per = f1_score(Y_dev_a, pred_per, average='macro')
log_experiment('E09', 'asp_threshold_0.5', 'Multi-label aspect fixed threshold 0.5', {'th': 0.5}, dev_f1=f1_05)
log_experiment('E09', 'asp_threshold_per_class', 'Multi-label aspect OOF per-class thresholds', {'ths': best_ths}, dev_f1=f1_per)

comp = []
for j, name in enumerate(ASPECT_GROUPS):
    comp.append({
        'aspect': name, 'tuned_th': best_ths[j],
        'f1_th_0.5': round(f1_score(Y_dev_a[:, j], pred_05[:, j]), 4),
        'f1_tuned': round(f1_score(Y_dev_a[:, j], pred_per[:, j]), 4),
    })
print(f'Dev Macro F1 -> fixed 0.5: {f1_05:.4f} | per-class tuned: {f1_per:.4f}')
pd.DataFrame(comp)

Per-aspect OOF tuned thresholds: {'PERF_FEAT': 0.4, 'SCREEN_CAM': 0.46, 'BATTERY': 0.42, 'PRICE_SER': 0.46, 'DESIGN_GEN': 0.3}


Dev Macro F1 -> fixed 0.5: 0.8911 | per-class tuned: 0.9047


,aspect,tuned_th,f1_th_0.5,f1_tuned
0,PERF_FEAT,0.40,0.9009,0.9046
1,SCREEN_CAM,0.46,0.8819,0.8886
2,BATTERY,0.42,0.9608,0.9673
3,PRICE_SER,0.46,0.8619,0.8674
4,DESIGN_GEN,0.30,0.8501,0.8958


## 3. KẾT LUẬN & HƯỚNG GIẢI QUYẾT ĐỀ XUẤT

- **Sentiment (3 lớp):**
  - Không cân bằng (`weight_none`) cho Accuracy cao (`0.7982`) nhưng lớp Trung lập bị bỏ sót nặng (`neu_R = 0.3287`, `neu_F1 = 0.4264`).
  - `oversample_neu` (nhân bản ngẫu nhiên lớp Trung lập trên train) đạt **Macro F1 cao nhất `0.7521`** (`neu_F1 = 0.5494`, Accuracy `0.7919`), nhỉnh hơn `weight_balanced` (`0.7478`). `weight_smooth` phù hợp khi muốn cân bằng Precision Trung lập (`0.5464`) và Accuracy (`0.7973`).
- **Multi-label Aspect (`asp_0..asp_4`):**
  - Tuning 5 ngưỡng riêng trên **out-of-fold train** ra bộ ngưỡng `{'PERF_FEAT': 0.40, 'SCREEN_CAM': 0.46, 'BATTERY': 0.42, 'PRICE_SER': 0.46, 'DESIGN_GEN': 0.30}` giúp **Dev Macro F1 tăng từ `0.8911` lên `0.9047` (`+0.0136`)**, riêng `DESIGN_GEN` tăng `+0.0457` (`0.8501 → 0.8958`).
- **Hướng giải quyết (cho NB03 / NB04):** áp dụng **ngưỡng riêng cho từng khía cạnh** (tune trên out-of-fold train cho baseline hoặc `df_calib` cho PhoBERT) thay cho ngưỡng cố định `0.5`, kết hợp `oversample_neu` hoặc `weight_balanced` cho Sentiment.
